<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.2: 组合逻辑
**上一节: [你的第一个 Chisel 模块](2.1_first_module.ipynb)**<br>
**下一节: [控制流](2.3_control_flow.ipynb)**

## 动机
在本节中，你将看到如何使用 Chisel 组件来实现组合逻辑。
我们将演示三种基本的 Chisel 类型：`UInt` - 无符号整数；`SInt` - 有符号整数，以及 `Bool` - 真或假，如何连接和操作。
请注意，所有 Chisel 变量都声明为 Scala `val`。
永远不要将 Scala `var` 用于硬件构造，因为构造本身一旦定义就永远不会改变；只有其值在运行硬件时可能改变。
导线（Wires）可用于参数化类型。

## 设置

In [1]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

Compiling /Users/zhaochanglong/Documents/chisel-bootcamp-zh/Main.sc

Downloaded https://repo1.maven.org/maven2/edu/berkeley/cs/chisel3_2.12/maven-metadata.xml
Downloaded https://repo1.maven.org/maven2/edu/berkeley/cs/chisel3_2.12/3.4.4/chisel3_2.12-3.4.4.pom
Downloaded https://repo1.maven.org/maven2/edu/berkeley/cs/chisel3-macros_2.12/3.4.4/chisel3-macros_2.12-3.4.4.pom
Downloaded https://repo1.maven.org/maven2/edu/berkeley/cs/chisel3-core_2.12/3.4.4/chisel3-core_2.12-3.4.4.pom
Downloaded https://repo1.maven.org/maven2/edu/berkeley/cs/firrtl_2.12/1.4.4/firrtl_2.12-1.4.4.pom
Downloaded https://repo1.maven.org/maven2/org/json4s/json4s-native_2.12/3.6.9/json4s-native_2.12-3.6.9.pom
Downloaded https://repo1.maven.org/maven2/com/google/protobuf/protobuf-java/3.9.0/protobuf-java-3.9.0.pom
Downloaded https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.7.1/antlr4-runtime-4.7.1.pom
Downloaded https://repo1.maven.org/maven2/net/jcazevedo/moultingyaml_2.12/0.4.2/moultingyaml_2.12-0.4.2.pom
Downloaded https://repo1.maven.org/maven2/org/apache/commons/commons-

Compiling /Users/zhaochanglong/Documents/chisel-bootcamp-zh/Main.sc #2

path: String = "/Users/zhaochanglong/Documents/chisel-bootcamp-zh/source/load-ivy.sc"

In [2]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

import chisel3._

import chisel3.util._

import chisel3.tester._

import chisel3.tester.RawTester.test

---
# 常见运算符
既然你理解了 `Module` 是如何构造的，让我们来制作一些硬件！看看下面的空模块。

In [ ]:
class MyModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })
}

我们将类命名为 `MyModule`，它扩展了 `Module`。这意味着它将被映射到 Verilog 中的硬件模块。我们的 `MyModule` 模块有一个输入和一个输出。输入是一个 4 位无符号整数（`UInt`），输出也是如此。

<span style="color:blue">**示例：Scala 和 Chisel 运算符看起来相同**</span><br>
让我们看看可以对数据执行的不同操作。

In [3]:
class MyModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })

  val two  = 1 + 1
  println(two)
  val utwo = 1.U + 1.U
  println(utwo)
  
  io.out := io.in
}
println(getVerilog(new MyModule))

Elaborating design...
2
UInt<1>(OpResult in MyModule)
Done elaborating.
module MyModule(
  input        clock,
  input        reset,
  input  [3:0] io_in,
  output [3:0] io_out
);
  assign io_out = io_in; // @[cmd2.sc 12:10]
endmodule



defined class MyModule

我们创建了两个 `val`。第一个将两个 Scala `Int` 相加，因此 `println` 打印出整数 2。第二个 `val` 将两个 *Chisel* `UInt` 相加，因此 `println` 将其视为硬件节点并打印出类型名称和指针（`chisel3.core.UInt@d`）。请注意，`1.U` 是从 Scala `Int`（1）到 Chisel `UInt` 字面量的类型转换。

我们需要驱动输出到某个东西，所以现在只是将其连接到输入，就像上一个教程中的直通模块一样。

<span style="color:blue">**示例：不兼容的操作**</span><br>
如果我们将 Chisel `1.U` 与字面量 `1` 相加会发生什么？这些类型不兼容，因为前者是值为 1 的硬件导线，而后者是值为 1 的 Scala 值。所以 Chisel 会给出类型不匹配错误。

In [3]:
class MyModuleTwo extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })

  val twotwo = 1.U + 1
  println(twotwo)
  
  io.out := io.in
}
println(getVerilog(new MyModuleTwo))

cmd3.sc:7: type mismatch;
 found   : Int(1)
 required: chisel3.UInt
  val twotwo = 1.U + 1
                     ^Compilation Failed

: 

在执行操作时记住类型之间的区别很重要。Scala 是一种强类型语言，因此任何类型转换都必须是显式的。

<span style="color:blue">**示例：更多 Chisel 运算符**</span><br>
其他常见操作包括减法和乘法。这些操作在无符号整数上的处理方式符合预期。让我们看看这些操作的实际应用。我们展示了 Verilog，尽管有一些底层的 Chisel 特性使简单的代码变得模糊。

In [4]:
class MyOperators extends Module {
  val io = IO(new Bundle {
    val in      = Input(UInt(4.W))
    val out_add = Output(UInt(4.W))
    val out_sub = Output(UInt(4.W))
    val out_mul = Output(UInt(4.W))
  })

  io.out_add := 1.U + 4.U
  io.out_sub := 2.U - 1.U
  io.out_mul := 4.U * 2.U
}
println(getVerilog(new MyOperators))

Elaborating design...
Done elaborating.
module MyOperators(
  input        clock,
  input        reset,
  input  [3:0] io_in,
  output [3:0] io_out_add,
  output [3:0] io_out_sub,
  output [3:0] io_out_mul
);
  wire [1:0] _T_3 = 2'h2 - 2'h1; // @[cmd3.sc 10:21]
  wire [4:0] _T_4 = 3'h4 * 2'h2; // @[cmd3.sc 11:21]
  assign io_out_add = 4'h5; // @[cmd3.sc 9:21]
  assign io_out_sub = {{2'd0}, _T_3}; // @[cmd3.sc 10:21]
  assign io_out_mul = _T_4[3:0]; // @[cmd3.sc 11:14]
endmodule



defined class MyOperators

这是上述操作的示例测试器。与上一个教程中使用匿名测试器类不同，我们将创建一个显式的测试器类。这只是编写测试器的另一种方式。

In [5]:
test(new MyOperators) {c =>
  c.io.out_add.expect(5.U)
  c.io.out_sub.expect(1.U)
  c.io.out_mul.expect(8.U)
}
println("SUCCESS!!")

Elaborating design...
Done elaborating.
test MyOperators Success: 0 tests passed in 2 cycles in 0.013959 seconds 143.28 Hz
SUCCESS!!


<span style="color:blue">**示例：多路选择器和连接**</span><br>
除了加法、减法和乘法之外，Chisel 还有多路选择器和连接运算符。这些如下所示。`Mux` 的操作类似于传统的三元运算符，顺序为（选择信号，为真时的值，为假时的值）。请注意，`true.B` 和 `false.B` 是创建 Chisel Bool 字面量的首选方式。`Cat` 的顺序是先 MSB 后 LSB（其中 B 指位或比特），并且只接受两个参数。连接两个以上的值需要多个 `Cat` 调用或将在后续章节中介绍的高级 Chisel 和 Scala 特性。

In [6]:
class MyOperatorsTwo extends Module {
  val io = IO(new Bundle {
    val in      = Input(UInt(4.W))
    val out_mux = Output(UInt(4.W))
    val out_cat = Output(UInt(4.W))
  })

  val s = true.B
  io.out_mux := Mux(s, 3.U, 0.U) // should return 3.U, since s is true
  io.out_cat := Cat(2.U, 1.U)    // concatenates 2 (b10) with 1 (b1) to give 5 (101)
}

println(getVerilog(new MyOperatorsTwo))

test(new MyOperatorsTwo) { c =>
  c.io.out_mux.expect(3.U)
  c.io.out_cat.expect(5.U)
}
println("SUCCESS!!")

Elaborating design...
Done elaborating.
module MyOperatorsTwo(
  input        clock,
  input        reset,
  input  [3:0] io_in,
  output [3:0] io_out_mux,
  output [3:0] io_out_cat
);
  assign io_out_mux = 4'h3; // @[cmd5.sc 9:20]
  assign io_out_cat = 4'h5; // @[Cat.scala 30:58]
endmodule

Elaborating design...
Done elaborating.
test MyOperatorsTwo Success: 0 tests passed in 2 cycles in 0.000848 seconds 2359.42 Hz
SUCCESS!!


defined class MyOperatorsTwo

请注意，Verilog 包含常量而不是实际的多路选择器或连接逻辑。这是因为 FIRRTL 转换已经简化了电路，消除了明显的逻辑。

有关更完整的 Chisel 运算符列表，请参阅 [Chisel 速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf)。有关最完整的运算符列表及其实现细节，请查看 [Chisel API](https://chisel-lang.org/api/latest/)。

---
# 练习
要完成这些练习，你可能需要查看 [Chisel 速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf)。

<span style="color:red">**练习：MAC**</span><br>
创建一个实现乘累加函数 `(A*B)+C` 并通过测试平台的 Chisel 模块。

In [7]:
class MAC extends Module {
  val io = IO(new Bundle {
    val in_a = Input(UInt(4.W))
    val in_b = Input(UInt(4.W))
    val in_c = Input(UInt(4.W))
    val out  = Output(UInt(8.W))
  })

  io.out := (io.in_a * io.in_b) + io.in_c
}

test(new MAC) { c =>
  val cycles = 100
  import scala.util.Random
  for (i <- 0 until cycles) {
    val in_a = Random.nextInt(16)
    val in_b = Random.nextInt(16)
    val in_c = Random.nextInt(16)
    c.io.in_a.poke(in_a.U)
    c.io.in_b.poke(in_b.U)
    c.io.in_c.poke(in_c.U)
    c.io.out.expect((in_a * in_b + in_c).U)
  }
}
println("SUCCESS!!")

Elaborating design...
Done elaborating.
test MAC Success: 0 tests passed in 2 cycles in 0.020255 seconds 98.74 Hz
SUCCESS!!


defined class MAC

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
class MAC extends Module {
  val io = IO(new Bundle {
    val in_a = Input(UInt(4.W))
    val in_b = Input(UInt(4.W))
    val in_c = Input(UInt(4.W))
    val out  = Output(UInt(8.W))
  })

  io.out := (io.in_a * io.in_b) + io.in_c
}
</pre></article></div></section></div>

<span style="color:red">**练习：仲裁器**</span><br>
以下电路将从 FIFO 进入的数据仲裁到两个并行处理单元。FIFO 和处理单元（PEs）通过就绪-有效接口进行通信。构建仲裁器以将数据发送到准备好接收数据的 PE，如果两个 PE 都准备好接收数据，则优先考虑 PE0。请记住，当至少有一个 PE 可以接收数据时，仲裁器应告诉 FIFO 它已准备好接收数据。此外，在断言数据有效之前，等待 PE 断言其已就绪。你可能需要二进制运算符来完成此练习。

<img src="images/arbiter.png" width="687" height="177">

In [8]:
class Arbiter extends Module {
  val io = IO(new Bundle {
    // FIFO
    val fifo_valid = Input(Bool())
    val fifo_ready = Output(Bool())
    val fifo_data  = Input(UInt(16.W))
    
    // PE0
    val pe0_valid  = Output(Bool())
    val pe0_ready  = Input(Bool())
    val pe0_data   = Output(UInt(16.W))
    
    // PE1
    val pe1_valid  = Output(Bool())
    val pe1_ready  = Input(Bool())
    val pe1_data   = Output(UInt(16.W))
  })
    io.fifo_ready := io.pe0_ready || io.pe1_ready
    io.pe0_valid := io.fifo_valid && io.pe0_ready
    io.pe1_valid := io.fifo_valid && !io.pe0_ready && io.pe1_ready
    io.pe0_data := io.fifo_data
    io.pe1_data := io.fifo_data
}

test(new Arbiter) { c =>
  import scala.util.Random
  val data = Random.nextInt(65536)
  c.io.fifo_data.poke(data.U)
  
  for (i <- 0 until 8) {
    c.io.fifo_valid.poke((((i >> 0) % 2) != 0).B)
    c.io.pe0_ready.poke((((i >> 1) % 2) != 0).B)
    c.io.pe1_ready.poke((((i >> 2) % 2) != 0).B)

    c.io.fifo_ready.expect((i > 1).B)
    c.io.pe0_valid.expect((i == 3 || i == 7).B)
    c.io.pe1_valid.expect((i == 5).B)
    
    if (i == 3 || i ==7) {
      c.io.pe0_data.expect((data).U)
    } else if (i == 5) {
      c.io.pe1_data.expect((data).U)
    }
  }
}
println("SUCCESS!!")

Elaborating design...
Done elaborating.
test Arbiter Success: 0 tests passed in 2 cycles in 0.004251 seconds 470.46 Hz
SUCCESS!!


defined class Arbiter

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  io.fifo_ready := io.pe0_ready || io.pe1_ready
  io.pe0_valid := io.fifo_valid && io.pe0_ready
  io.pe1_valid := io.fifo_valid && io.pe1_ready && !io.pe0_ready
  io.pe0_data := io.fifo_data
  io.pe1_data := io.fifo_data
</pre></article></div></section></div>

<span style="color:red">**练习：参数化加法器（可选）**</span><br>
这个可选练习向你展示了 Chisel 最强大的功能之一：参数化能力。为了演示这一点，我们将构建一个参数化加法器，它可以在发生溢出时饱和输出，或者截断结果（即环绕）。

首先，看下面的 `Module`。我们传递给它的参数称为 `saturate`，类型为 *Scala* `Boolean`。这不是 Chisel `Bool`。因此，我们不是在创建一个可以饱和或截断的单个硬件加法器，而是在创建一个 *生成器*，它可以产生饱和硬件加法器 *或* 截断硬件加法器。这个决定是在编译时做出的。

接下来，注意输入和输出都是 4 位 `UInt`。Chisel 具有内置的宽度推断功能，如果你查看 [速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf)，你会看到正常求和的位宽等于两个输入的最大位宽。这意味着

```scala
val sum = io.in_a + io.in_b
```

将使 `sum` 成为 4 位导线，并且该值将是 4 位输入的截断结果。要检查求和是否应该饱和，你需要将结果放在 5 位导线中。这可以通过 `+&` 求和来完成，如速查表中所示。

```scala
val sum = io.in_a +& io.in_b
```

最后，请注意，将 4 位 `UInt` 导线连接到 5 位 `UInt` 导线默认会截断 MSB。你可以使用这个来轻松截断非饱和加法器的 5 位和。

In [12]:
class ParameterizedAdder(saturate: Boolean) extends Module {
  val io = IO(new Bundle {
    val in_a = Input(UInt(4.W))
    val in_b = Input(UInt(4.W))
    val out  = Output(UInt(4.W))
  })

  val sum = io.in_a +& io.in_b
  if (saturate) {
    io.out := Mux(sum > 15.U, 15.U, sum)
  } else {
    io.out := sum
  }
}

for (saturate <- Seq(true, false)) {
  test(new ParameterizedAdder(saturate)) { c =>
    // 100 random tests
    val cycles = 100
    import scala.util.Random
    import scala.math.min
    for (i <- 0 until cycles) {
      val in_a = Random.nextInt(16)
      val in_b = Random.nextInt(16)
      c.io.in_a.poke(in_a.U)
      c.io.in_b.poke(in_b.U)
      if (saturate) {
        c.io.out.expect(min(in_a + in_b, 15).U)
      } else {
        c.io.out.expect(((in_a + in_b) % 16).U)
      }
    }
    
    // ensure we test saturation vs. truncation
    c.io.in_a.poke(15.U)
    c.io.in_b.poke(15.U)
    if (saturate) {
      c.io.out.expect(15.U)
    } else {
      c.io.out.expect(14.U)
    }
  }
}
println("SUCCESS!!")

Elaborating design...
Done elaborating.
test ParameterizedAdder Success: 0 tests passed in 2 cycles in 0.004184 seconds 478.04 Hz
Elaborating design...
Done elaborating.
test ParameterizedAdder Success: 0 tests passed in 2 cycles in 0.002393 seconds 835.71 Hz
SUCCESS!!


defined class ParameterizedAdder

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-3" />
<label for="check-3"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val sum = io.in_a +& io.in_b
  if (saturate) {
    io.out := Mux(sum > 15.U, 15.U, sum)
  } else {
    io.out := sum
  }
</pre></article></div></section></div>

---
# 完成！

[返回顶部。](#top)